
# AGN X-ray coronae: luminosity sequence and spectral hardness at high energies

AGN coronae are compact hot regions where the hard X-ray power law (photon
index ~1.7–2.0) is produced via Compton scattering off hot electrons.
The X-ray spectrum reflects the coronal temperature, optical depth, and
geometry. This demo varies L_bol across six decades (10⁴²–10⁴⁶·⁵ erg/s) to
show the gradual brightening of the X-ray continuum and the persistence of
the power-law form across the luminosity sequence. A separate panel isolates
key spectral features: soft excess (0.5–2 keV), hard continuum (2–10 keV),
Compton reflection hump (10–100 keV), and the iron K-α line (6.4 keV).

Reference: Wilkins et al. 2020, MNRAS, 493, 5548 (AGN X-ray corona models).


In [ ]:
import warnings

import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from tengri.analysis.plotting import setup_style
from tengri.xray import xray_agn_corona

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

wavelength = jnp.logspace(np.log10(0.0124), np.log10(124.0), 512)
wave_keV = 12.398 / np.array(wavelength)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel 1: Luminosity sequence (discrete)
ax = axes[0, 0]
log_lbol_vals = np.array([43.0, 44.0, 45.0, 46.0])
for log_lbol in log_lbol_vals:
    l_xray = xray_agn_corona(wavelength, L_agn_bol=10.0**log_lbol)
    ax.loglog(wave_keV, np.array(l_xray), lw=1.4, label=r"$\log L_{\rm bol}=" + f"{log_lbol:.0f}$")

ax.set_xlim(0.1, 1000)
ax.set_ylim(1e22, 1e27)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")
ax.text(
    0.05,
    0.95,
    "a) Luminosity sequence",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=10,
)
ax.legend(fontsize=9, frameon=False, loc="lower left")

# Panel 2: Spectral features
ax = axes[0, 1]
log_lbol = 44.0
l_xray = xray_agn_corona(wavelength, L_agn_bol=10.0**log_lbol)
ax.loglog(wave_keV, np.array(l_xray), "C0-", lw=2.0)

ax.axvspan(0.5, 2.0, alpha=0.15, color="C1")
ax.axvspan(2.0, 10.0, alpha=0.15, color="C2")
ax.axvspan(10.0, 100.0, alpha=0.15, color="C3")
ax.axvline(6.4, color="red", ls="--", lw=1.0, alpha=0.5)

ax.set_xlim(0.1, 1000)
ax.set_ylim(1e22, 1e27)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")
ax.text(
    0.05,
    0.95,
    "b) Key spectral features",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=10,
)
ax.text(1.2, 0.55e22, "soft excess\n(0.5–2 keV)", fontsize=8, color="C1")
ax.text(5.0, 0.55e22, "hard PL\n(2–10 keV)", fontsize=8, color="C2")
ax.text(50.0, 0.55e22, "reflection\n(>10 keV)", fontsize=8, color="C3")

# Panel 3: Intermediate luminosity range
ax = axes[1, 0]
log_lbol_mid = np.array([45.0, 45.5, 46.0, 46.5])
for log_lbol in log_lbol_mid:
    l_xray = xray_agn_corona(wavelength, L_agn_bol=10.0**log_lbol)
    ax.loglog(wave_keV, np.array(l_xray), lw=1.4, label=r"$\log L_{\rm bol}=" + f"{log_lbol:.1f}$")

ax.set_xlim(0.1, 1000)
ax.set_ylim(1e22, 1e27)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")
ax.text(
    0.05,
    0.95,
    "c) Ultra-luminous range",
    transform=ax.transAxes,
    verticalalignment="top",
    fontsize=10,
)
ax.legend(fontsize=9, frameon=False, loc="lower left")

# Panel 4: Full luminosity continuum
ax = axes[1, 1]
log_lbol_range = np.linspace(42.0, 46.5, 12)
norm = mpl.colors.Normalize(vmin=log_lbol_range.min(), vmax=log_lbol_range.max())
cmap = plt.get_cmap("viridis")

for log_lbol in log_lbol_range:
    l_xray = xray_agn_corona(wavelength, L_agn_bol=10.0**log_lbol)
    mask = np.array(l_xray) > 0
    ax.loglog(
        wave_keV[mask], np.array(l_xray)[mask], lw=1.0, color=cmap(norm(log_lbol)), alpha=0.7
    )

ax.set_xlim(0.1, 1000)
ax.set_ylim(1e22, 1e27)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]")
ax.text(
    0.05, 0.95, "d) Full SED family", transform=ax.transAxes, verticalalignment="top", fontsize=10
)

cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.01)
cbar.set_label(r"$\log(L_{\rm bol})$ [erg s$^{-1}$]")

fig.tight_layout()
fig.savefig("plot_xray_agn.png", dpi=150, bbox_inches="tight")